In [6]:
!pip install --upgrade google-cloud-bigquery pandas pyarrow db-dtypes

!pip install db-dtypes


Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Defaulting to user installation because normal site-packages is not writeable


In [1]:
from google.cloud import bigquery
import pandas as pd
import os

# Point to your downloaded JSON key
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./mke-trees-6296ce9c1929.json"

client = bigquery.Client()
database = "mke-trees.mke_tree_dataset."
tree_table = database + "tree"
address_table = database + "address"
quarter_section_table = database + "quarter_section"
service_requst_table = database + "service_request"
species_table = database + "species"
tables = [tree_table, address_table, quarter_section_table, service_requst_table, species_table]


In [4]:
import pandas as pd

df = pd.read_csv("./raw-data/full_tree_inventory.csv", encoding="ISO-8859-1")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: './raw-data/full_tree_inventory.csv'

In [16]:
# Map DataFrame columns to BigQuery `tree` table columns
# df columns (source) -> tree table columns (destination)
tree_column_mapping = {
    "site_id": "site_id",
    "inspect_dt": "inspection_date",
    "inspect_tm": "inspection_time",
    "notes": "notes",
    "inv_date": "inventory_date",
    "Rating": "rating",
    "DBH": "dbh",
    "CrownWd": "crown_width",
    "Height": "height",
    "Growspace": "grow_space",
    "Damage": "damage",
    "Insects": "insect",
    "Disease": "disease",
    "Status": "status",
    "x": "x",
    "y": "y",
    "latitude": "latitude",
    "longitude": "longitude",
}

# Build DataFrame that matches the `tree` table schema for the mapped columns
tree_df = df[list(tree_column_mapping.keys())].rename(columns=tree_column_mapping)

# Basic type coercions for BigQuery (dates and times)
# Coerce dates
for date_col in ["inspection_date", "inventory_date"]:
    tree_df[date_col] = pd.to_datetime(tree_df[date_col], errors="coerce").dt.date

# Coerce times
if "inspection_time" in tree_df.columns:
    tree_df["inspection_time"] = pd.to_datetime(
        tree_df["inspection_time"], errors="coerce"
    ).dt.time

# Load into BigQuery `tree` table
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # change to WRITE_TRUNCATE if you want to overwrite
)

load_job = client.load_table_from_dataframe(tree_df, tree_table, job_config=job_config)
load_job.result()  # Waits for job to complete

print(f"Loaded {load_job.output_rows} rows into {tree_table}.")

C:\Users\edex\AppData\Local\Temp\ipykernel_19696\1485087682.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tree_df["inspection_time"] = pd.to_datetime(
c:\Users\edex\AppData\Local\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 205901 rows into mke-trees.mke_tree_dataset.tree.


In [14]:
# Change `dbh` column type from INTEGER to FLOAT64 in the `tree` table
# BigQuery doesn't support in-place type changes, so we recreate the table
# with the same data but `dbh` cast to FLOAT64.

from google.cloud import bigquery

sql = f"""
CREATE OR REPLACE TABLE `{tree_table}` AS
SELECT
  site_id,
  address_id,
  quarter_section,
  species_id,
  last_changed,
  inspection_date,
  inspection_time,
  notes,
  inventory_date,
  rating,
  CAST(dbh AS FLOAT64) AS dbh,
  crown_width,
  crown_asymmetry,
  height,
  grow_space,
  damage,
  insect,
  disease,
  canker,
  status,
  x,
  y,
  latitude,
  longitude,
  building_distance,
  car_distance,
  road_distance,
  sidewalk_distance,
  driveway_distance
FROM `{tree_table}`;
"""

job = client.query(sql)
job.result()  # wait for completion

print("Updated `dbh` column to FLOAT64 in BigQuery table:", tree_table)

Updated `dbh` column to FLOAT64 in BigQuery table: mke-trees.mke_tree_dataset.tree


In [2]:
query = f"SELECT * FROM `{tree_table}` LIMIT 100"
df_trees = client.query(query).to_dataframe()
df_trees.head()


/home/ad.msoe.edu/edex/.local/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,site_id,inspection_date,inspection_time,notes,inventory_date,rating,dbh,crown_width,height,grow_space,damage,insect,disease,status,x,y,latitude,longitude
0,5232,2022-01-28,15:02:56,None,2010-01-25,Good,22.0,36,35,6.0,No,No,No,Tree,2496126.299,421461.0711,43.139209,-88.022963
1,5233,2022-01-28,15:12:04,None,2010-01-25,Good,19.0,25,30,6.0,No,No,No,Tree,2496123.448,421579.2586,43.139533,-88.022964
2,5234,2022-08-10,16:15:46,None,2010-01-25,Good,9.0,15,25,6.0,No,No,No,Tree,2496163.138,421577.6390,43.139526,-88.022815
3,5235,2022-01-28,15:13:34,1st n/of driveway,2010-01-25,Good,9.0,6,20,6.0,No,No,No,Tree,2496137.037,421704.2361,43.139875,-88.022902
4,5236,2023-11-02,11:18:20,By south lot line,2022-01-28,Good,2.0,0,0,6.0,No,No,No,Tree,2496171.590,421695.6920,43.139850,-88.022773
